# Talking to an MCP Server from LangChain

Connecting to the local MCP server defined in [`4.mcp_server_basic.py`](4.mcp_server_basic.py) and using its tools inside a LangChain agent.

> **Before you start:** run the server in a **separate terminal** (see Step 1). This notebook is
> just the **client** — it connects to the already-running server over HTTP.

**Main Steps:**
1. Connect to the running MCP server (started from a terminal).
2. Discover the tools it exposes and inspect their schemas.
3. Call a tool directly.
4. Hand the MCP tools to a LangChain agent and let the LLM decide when to use them.
5. Read an MCP **resource** and the server's default **prompt**.

> **Why HTTP instead of `stdio`?** The `stdio` transport makes the *client* launch the server as a
> child process. On **Windows**, a Jupyter kernel runs a `SelectorEventLoop` that **cannot spawn
> subprocesses**, so `stdio` fails with `NotImplementedError` / `UnsupportedOperation: fileno`.
> Running the server over **HTTP** avoids this completely — the notebook is just an HTTP client.
> (This is a different problem from `asyncio.run()` re-entrancy; `nest_asyncio` does **not** fix it.)


In [ ]:
%run ../langchain_common.py

## Step 1 — Start the server (in a terminal), then connect

**1. Start the server** in a **separate terminal**, from the `wk4_langchain_agents_mcp` folder.
Pass the `--http` flag so it runs over HTTP:

```powershell
python 4.mcp_server_basic.py --http
```

You should see a log line like:
`... Launching MCP server over streamable-http at http://127.0.0.1:8000/mcp`.
Leave that terminal running while you work through this notebook (press **Ctrl+C** there to stop it later).

> Without `--http`, the server defaults to the `stdio` transport (for a client that launches it as a
> subprocess) and won't open an HTTP port — so this notebook wouldn't be able to connect.

**2. Connect** from this notebook by pointing `MultiServerMCPClient` at the server's URL
(`transport="streamable_http"`). The notebook never spawns a process — it's purely a client.


In [ ]:
import socket
from langchain_mcp_adapters.client import MultiServerMCPClient

# Must match the host/port the server is listening on (these are the defaults).
MCP_HOST, MCP_PORT = "127.0.0.1", 8000
MCP_URL = f"http://{MCP_HOST}:{MCP_PORT}/mcp"

# Quick sanity check: is the server actually running? (Helpful reminder if you skipped Step 1.)
with socket.socket() as s:
    s.settimeout(1.0)
    if s.connect_ex((MCP_HOST, MCP_PORT)) != 0:
        raise RuntimeError(
            f"No MCP server found at {MCP_URL}. Start it in a terminal first:\n"
            "    python 4.mcp_server_basic.py --http"
        )

# Point the client at the already-running server over HTTP.
client = MultiServerMCPClient(
    {
        "mcp_server": {
            "url": MCP_URL,
            "transport": "streamable_http",
        }
    }
)
print(f"Client configured to connect to {MCP_URL}")


## Step 2 — Discover the server's tools

`get_tools()` is **async**, so we `await` it. Under the hood it connects to the running HTTP
server, runs the MCP handshake, asks for the tool list, and wraps each one as a LangChain
`BaseTool` you can use anywhere a normal tool is expected.

In [ ]:
tools = await client.get_tools()

print(f"Discovered {len(tools)} tool(s):")
for t in tools:
    print(f"  - {t.name}: {t.description.strip().splitlines()[0]}")

### Inspect a tool's input schema

Each MCP tool advertises a JSON schema for its arguments. LangChain exposes this as
`args_schema`, which is exactly what the LLM sees when deciding how to call the tool.

In [ ]:
search_tool = next(t for t in tools if t.name == "search_web")

pp({
    "name": search_tool.name,
    "description": search_tool.description,
    "args": search_tool.args,
})

## Step 3 — Call the tool from the MCP Server

In [ ]:
tool_msg = await search_tool.ainvoke(
    {
        "type": "tool_call",
        "name": search_tool.name,
        "args": {"query": "What is the Model Context Protocol (MCP)?"},
        "id": "1",
    }
)

data = tool_msg.artifact["structured_content"]["result"]
for item in data.get("results", [])[:3]:
    print("-", item.get("title"))
    print("  ", item.get("url"))


## Step 4 — Give the tools to a LangChain agent

In [ ]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You are a helpful research assistant. Use the search_web tool to find current "
        "information when needed, then answer the user's question concisely with sources."
    ),
)

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="In one paragraph, what is the Model Context Protocol and who introduced it?")]}
)

# Some models return content as a list of blocks (reasoning + text); grab the answer text.
final = response["messages"][-1].content

pp(final)


In [ ]:
for m in response["messages"]:
    m.pretty_print()

## Step 5 MCP Server - resource and the default prompt

In [ ]:
from langchain_mcp_adapters.resources import load_mcp_resources
from langchain_mcp_adapters.prompts import load_mcp_prompt

async with client.session("mcp_server") as session:
    # List + load resources exposed by the server.
    resources = await load_mcp_resources(session)
    print(f"Resources: {len(resources)}")
    for r in resources:
        pp(r.as_string())

In [ ]:
from langchain_mcp_adapters.resources import load_mcp_resources
from langchain_mcp_adapters.prompts import load_mcp_prompt

async with client.session("mcp_server") as session:
    # Load the server's default prompt as LangChain messages.
    prompt_messages = await load_mcp_prompt(session, "prompt")
    print("\nDefault prompt (first 200 chars):")
    print(prompt_messages[0].content.strip()[:200])